# Imports

In [10]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying loss

## load data

In [11]:
root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/results_ML3/diff_meta'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle', 'task_name']
metric_names = ['mse', 'mae', 'cov']

df = []
for exp_dir in exp_dirs:
    if 'MAE' in exp_dir:
        continue
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    if len(metric) == 6:
        log_metrics = load_metric_from_log(os.path.join(exp_dir, 'result_long_term_forecast.txt'))
        cov_loss = log_metrics['cov']
        result.loc[:, metric_names] = metric[1], metric[0], cov_loss
    else:
        result.loc[:, metric_names] = metric[1], metric[0], metric[2]
    result.loc[:, ['meta_type']] = config[['meta_type']] if 'meta_type' in config.columns else 'all'
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/varying_meta.csv", index=False)

df.head(4)

KeyboardInterrupt: 

## pre-load

In [12]:
stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
log_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'
baselines = pd.read_csv(f'{log_root}/baselines_chosen.csv')
finetunes_all = pd.read_csv(f'{stats_root}/finetune_all_results.csv')
df = pd.read_csv(f'{stats_root}/varying_meta.csv')

datasets = ['ECL', 'Weather']
base = baselines.copy()
base = base[
    (base.data_id.isin(datasets) & (base.model == 'TQNet'))
]
base['meta_method'] = 'None'

finetunes = finetunes_all.copy()
finetunes = finetunes[(finetunes.first_order == 1) & finetunes.data_id.isin(datasets) & (finetunes.model == 'TQNet')]
finetunes['meta_method'] = 'FOMAML'

df.rename(columns={'task_name': 'meta_method'}, inplace=True)
df['meta_method'] = df['meta_method'].apply(lambda x: x.split('_meta_')[-1])
df['meta_method'] = df['meta_method'].replace({'imaml': 'iMAML', 'ml3': 'MAML', 'mamlpp': 'MAML++', 'reptile': 'Reptile'})

df = df[finetunes.columns]

finetunes = pd.concat([finetunes, df], ignore_index=True).reset_index(drop=True)
finetunes.head()

,model,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,mse,mae,cov,meta_type,exp_dir,meta_method
0,TQNet,96,ECL,0.002,0.002,0.02,1.0,0.0,0.0,type1,30,5,16,64,200,1,0.0,3,5.0,MSE,1,0.0,168,0.136294,0.230567,0.110568,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...,FOMAML
1,TQNet,96,ECL,0.002,0.002,0.02,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,MSE,1,0.0,168,0.136301,0.230566,0.108213,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...,FOMAML
2,TQNet,96,ECL,0.002,0.002,0.02,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,MSE,1,0.0,168,0.136555,0.230776,0.132053,diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...,FOMAML
3,TQNet,96,ECL,0.002,0.002,0.02,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,MSE,1,0.0,168,0.136288,0.230553,0.117994,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...,FOMAML
4,TQNet,96,ECL,0.002,0.002,0.02,1.0,0.0,0.0,type1,30,5,16,64,200,2,0.0,3,5.0,MSE,1,0.0,168,0.136283,0.230535,0.110568,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...,FOMAML


## pre-analysis

In [61]:

finetunes[(finetunes.model == 'TimeBridge') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'EMD')][columns].sort_values(by='mse').groupby(['learning_rate', 'auxi_lambda', 'batch_size', 'var_weight'])['pred_len'].nunique()

learning_rate  auxi_lambda  batch_size  var_weight
0.0002         0.01         16          0.002         4
               0.05         16          0.002         4
               0.10         16          0.002         4
0.0005         0.01         16          0.002         4
               0.05         16          0.002         4
               0.10         16          0.002         4
0.0010         0.01         16          0.002         4
               0.05         16          0.002         4
               0.10         16          0.002         4
Name: pred_len, dtype: int64

In [62]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'lradj', 'train_epochs', 'patience', 'mask_factor', 'meta_method', 'joint_forecast', 'ot_type', 'batch_size', 'auxi_loss', 'reg_sk', 'eps', 'var_weight', 'mean_weight']

# finetunes[(finetunes.model == 'TimeBridge') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'MMD@Linear')]

finetunes[(finetunes.model == 'TimeBridge') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'EMD') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)
# finetunes[(finetunes.model == 'TimeBridge') & (finetunes.data_id == 'ETTh1') & (finetunes.meta_method == 'MMD@Linear') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)
# finetunes[(finetunes.model == 'TimeBridge') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'KL') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)
# finetunes[(finetunes.model == 'TimeBridge') & (finetunes.data_id == 'Weather') & (finetunes.meta_method == 'KL') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)


# finetunes[(finetunes.model == 'Fredformer') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'EMD') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)
# finetunes[(finetunes.model == 'Fredformer') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'MMD@RBF') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)
# finetunes[(finetunes.model == 'Fredformer') & (finetunes.data_id == 'ECL') & (finetunes.meta_method == 'KL') & (finetunes.pred_len == 96)][columns].sort_values(by='mse').head(10)

,model,pred_len,data_id,mse,mae,learning_rate,rec_lambda,auxi_lambda,lradj,train_epochs,patience,mask_factor,distance,joint_forecast,ot_type,batch_size,auxi_loss,reg_sk,eps,var_weight,mean_weight
7812,TimeBridge,96,ECL,0.138198,0.235461,0.0010,0.99,0.01,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7824,TimeBridge,96,ECL,0.138337,0.235763,0.0010,0.90,0.10,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7800,TimeBridge,96,ECL,0.138963,0.235938,0.0010,0.95,0.05,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7796,TimeBridge,96,ECL,0.141079,0.238766,0.0005,0.95,0.05,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7808,TimeBridge,96,ECL,0.141379,0.238867,0.0005,0.99,0.01,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7820,TimeBridge,96,ECL,0.142296,0.239229,0.0005,0.90,0.10,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7804,TimeBridge,96,ECL,0.146384,0.241695,0.0002,0.99,0.01,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7792,TimeBridge,96,ECL,0.146408,0.241657,0.0002,0.95,0.05,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0
7816,TimeBridge,96,ECL,0.146445,0.241616,0.0002,0.90,0.10,type1,10,3,0.0,EMD,1,upper_bound,16,NaN,0.005,1.000000e-09,0.002,1.0


## preprocess

In [13]:
df1 = base.copy()
df2 = finetunes.copy()

# df2_MMD_Linear_TB_h1 = df2[(df2.meta_method == 'MMD@Linear') & (df2.model == 'TimeBridge') & (df2.data_id == 'ETTh1') & (df2.learning_rate == 0.0005) & (df2.auxi_lambda == 0.1)]
# df2_EMD_TB_ecl = df2[(df2.meta_method == 'EMD') & (df2.model == 'TimeBridge') & (df2.data_id == 'ECL') & (df2.learning_rate == 0.0005) & (df2.auxi_lambda == 0.05)]
# df2_KL_TB_ecl = df2[(df2.meta_method == 'KL') & (df2.model == 'TimeBridge') & (df2.data_id == 'ECL') & (df2.learning_rate == 0.0005) & (df2.auxi_lambda == 0.05)]
# df2_KL_TB_wea = df2[(df2.meta_method == 'KL') & (df2.model == 'TimeBridge') & (df2.data_id == 'Weather') & (df2.learning_rate == 0.0012) & (df2.auxi_lambda == 0.005)]

# df2_EMD_FF_ecl = df2[(df2.meta_method == 'EMD') & (df2.model == 'Fredformer') & (df2.data_id == 'ECL') & (df2.learning_rate == 0.005) & (df2.auxi_lambda == 0.05)]
# df2_MMD_RBF_FF_ecl = df2[(df2.meta_method == 'MMD@RBF') & (df2.model == 'Fredformer') & (df2.data_id == 'ECL') & (df2.learning_rate == 0.005) & (df2.auxi_lambda == 0.05)]
# df2_KL_FF_ecl = df2[(df2.meta_method == 'KL') & (df2.model == 'Fredformer') & (df2.data_id == 'ECL') & (df2.learning_rate == 0.005) & (df2.auxi_lambda == 0.01)]


# df2_other_tb = df2[df2.model == 'TimeBridge']
# df2_other_tb = df2_other_tb[
#     (df2_other_tb.meta_method.isin(['DistDF', 'MMD@RBF', 'DF'])) |
#     ((df2_other_tb.meta_method == 'MMD@Linear') & (df2_other_tb.data_id.isin(['ETTm1', 'ETTm2', 'ETTh2', 'ECL', 'Weather']))) |
#     ((df2_other_tb.meta_method == 'KL') & (df2_other_tb.data_id.isin(['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2']))) |
#     ((df2_other_tb.meta_method == 'EMD') & (df2_other_tb.data_id.isin(['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'Weather'])))
# ]
# df2_other_ff = df2[df2.model == 'Fredformer']
# df2_other_ff = df2_other_ff[
#     (df2_other_ff.meta_method.isin(['DistDF', 'MMD@Linear', 'DF'])) |
#     ((df2_other_ff.meta_method == 'EMD') & (df2_other_ff.data_id.isin(['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'Weather']))) |
#     ((df2_other_ff.meta_method == 'MMD@RBF') & (df2_other_ff.data_id.isin(['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'Weather']))) |
#     ((df2_other_ff.meta_method == 'KL') & (df2_other_ff.data_id.isin(['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'Weather'])))
# ]
# df2 = pd.concat([
#     df2_MMD_Linear_TB_h1, 
#     df2_EMD_TB_ecl,
#     df2_KL_TB_ecl, df2_KL_TB_wea, 
#     df2_EMD_FF_ecl,
#     df2_MMD_RBF_FF_ecl, 
#     df2_KL_FF_ecl,
#     df2_other_tb, 
#     df2_other_ff
# ], ignore_index=True)

min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len', 'meta_method'])['mse'].idxmin()
df2 = df2.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'meta_method', 'mse', 'mae']
df_meta = pd.concat([df1[columns], df2[columns]], ignore_index=True)

dst_order = ['ECL', 'Weather']
df_meta['data_id'] = pd.Categorical(df_meta['data_id'], categories=dst_order, ordered=True)

model_order = ['TQNet']
df_meta['model'] = pd.Categorical(df_meta['model'], categories=model_order, ordered=True)

meta_method_order = ['None', 'iMAML', 'MAML', 'MAML++', 'Reptile', 'FOMAML']
df_meta['meta_method'] = pd.Categorical(df_meta['meta_method'], categories=meta_method_order, ordered=True)

df_meta_avg = df_meta.groupby(['model', 'data_id', 'meta_method']).mean(numeric_only=True).reset_index()
df_meta_avg['pred_len'] = 'Avg'
df_meta = pd.concat([df_meta, df_meta_avg]).reset_index(drop=True)

df_meta.sort_values(by=['model', 'data_id', 'meta_method', 'pred_len'], inplace=True)
df_meta.dropna(inplace=True, thresh=6)

# save_root = '/data/home/Licheng/workspace/TSF-PCA/stats'
# save_cols = ['data_id', 'pred_len', 'mse', 'mae', 'auxi_mode']
# df_meta.round(3)[save_cols].to_csv(f'{save_root}/diff_loss.csv', index=False, float_format='%.3f')
df_meta_avg.head(12)

/tmp/ipykernel_1705335/1286084741.py:54: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_meta_avg = df_meta.groupby(['model', 'data_id', 'meta_method']).mean(numeric_only=True).reset_index()


,model,data_id,meta_method,pred_len,mse,mae
0,TQNet,ECL,None,Avg,0.174991,0.265458
1,TQNet,ECL,iMAML,Avg,0.165994,0.257713
2,TQNet,ECL,MAML,Avg,0.166127,0.257944
3,TQNet,ECL,MAML++,Avg,0.165643,0.257345
4,TQNet,ECL,Reptile,Avg,0.167017,0.258753
5,TQNet,ECL,FOMAML,Avg,0.164592,0.256603
6,TQNet,Weather,None,Avg,0.245663,0.270328
7,TQNet,Weather,iMAML,Avg,0.244624,0.269983
8,TQNet,Weather,MAML,Avg,0.243269,0.269016
9,TQNet,Weather,MAML++,Avg,0.243278,0.268683


In [9]:
# 更简洁的实现
def calculate_gains_v2(df):
    result_df = df.copy()
    
    # 使用transform来获取每个data_id组内None方法的基准值
    baseline_mse = df.groupby('data_id')['mse'].transform(
        lambda x: x[df.loc[x.index, 'meta_method'] == 'None'].iloc[0] if any(df.loc[x.index, 'meta_method'] == 'None') else None
    )
    baseline_mae = df.groupby('data_id')['mae'].transform(
        lambda x: x[df.loc[x.index, 'meta_method'] == 'None'].iloc[0] if any(df.loc[x.index, 'meta_method'] == 'None') else None
    )
    
    # 计算增益
    result_df['mse_gain'] = (baseline_mse - df['mse']) / baseline_mse
    result_df['mae_gain'] = (baseline_mae - df['mae']) / baseline_mae
    
    return result_df

df_with_gains = calculate_gains_v2(df_meta_avg)
df_with_gains

/tmp/ipykernel_3978335/1550362871.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  baseline_mse = df.groupby('data_id')['mse'].transform(
/tmp/ipykernel_3978335/1550362871.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  baseline_mae = df.groupby('data_id')['mae'].transform(


,model,data_id,meta_method,pred_len,mse,mae,mse_gain,mae_gain
0,TQNet,ECL,None,Avg,0.174991,0.265458,0.000000,0.000000
1,TQNet,ECL,iMAML,Avg,0.165994,0.257713,0.051413,0.029177
2,TQNet,ECL,MAML,Avg,0.166127,0.257944,0.050654,0.028306
3,TQNet,ECL,MAML++,Avg,0.165643,0.257345,0.053423,0.030565
4,TQNet,ECL,Reptile,Avg,0.167017,0.258753,0.045566,0.025258
5,TQNet,ECL,FOMAML,Avg,0.164779,0.256880,0.058358,0.032314
6,TQNet,Weather,None,Avg,0.245663,0.270328,0.000000,0.000000
7,TQNet,Weather,iMAML,Avg,0.244624,0.269983,0.004230,0.001278
8,TQNet,Weather,MAML,Avg,0.243269,0.269016,0.009744,0.004854
9,TQNet,Weather,MAML++,Avg,0.243278,0.268683,0.009711,0.006085


## write to table

In [15]:
contents = []

for meta_method in ['None', 'iMAML', 'MAML', 'MAML++', 'Reptile', 'FOMAML']:
    line = r"\multicolumn{2}{l|}{" + meta_method + "}"
    contents.append(line)
    _df = df_with_gains[df_with_gains.meta_method == meta_method]
    line = ''
    for i, row in enumerate(_df.itertuples()):
        line += f"& {row.mse:.3f} & {row.mse_gain*100:.2f}\\% & {row.mae:.3f} & {row.mae_gain*100:.2f}\\% "
    line += r"\\"
    contents.append(line)

print('\n'.join(contents))

\multicolumn{2}{l|}{None}
& 0.175 & 0.00\% & 0.265 & 0.00\% & 0.246 & 0.00\% & 0.270 & 0.00\% \\
\multicolumn{2}{l|}{iMAML}
& 0.166 & 5.14\% & 0.258 & 2.92\% & 0.245 & 0.42\% & 0.270 & 0.13\% \\
\multicolumn{2}{l|}{MAML}
& 0.166 & 5.07\% & 0.258 & 2.83\% & 0.243 & 0.97\% & 0.269 & 0.49\% \\
\multicolumn{2}{l|}{MAML++}
& 0.166 & 5.34\% & 0.257 & 3.06\% & 0.243 & 0.97\% & 0.269 & 0.61\% \\
\multicolumn{2}{l|}{Reptile}
& 0.167 & 4.56\% & 0.259 & 2.53\% & 0.243 & 0.96\% & 0.269 & 0.43\% \\
\multicolumn{2}{l|}{FOMAML}
& 0.165 & 5.84\% & 0.257 & 3.23\% & 0.242 & 1.29\% & 0.268 & 0.95\% \\


In [65]:
import pandas as pd

# 为每个model和data_id组合找到最小和次最小的MSE和MAE
def find_best_and_second_best(df):
    results = {}
    
    for model in df['model'].unique():
        results[model] = {}
        for data_id in df['data_id'].unique():
            _df = df[(df['model'] == model) & (df['data_id'] == data_id)]
            if len(_df) > 0:
                # 找MSE最小和次最小
                mse_sorted = _df['mse'].sort_values()
                mse_best = mse_sorted.iloc[0] if len(mse_sorted) > 0 else None
                mse_second_best = mse_sorted.iloc[1] if len(mse_sorted) > 1 else None
                
                # 找MAE最小和次最小
                mae_sorted = _df['mae'].sort_values()
                mae_best = mae_sorted.iloc[0] if len(mae_sorted) > 0 else None
                mae_second_best = mae_sorted.iloc[1] if len(mae_sorted) > 1 else None
                
                results[model][data_id] = {
                    'mse_best': mse_best,
                    'mse_second_best': mse_second_best,
                    'mae_best': mae_best,
                    'mae_second_best': mae_second_best
                }
    
    return results

# 格式化数值
def format_value(value, best_val, second_best_val):
    if best_val is not None and abs(value - best_val) < 1e-6:  # 最小值
        return f"\\bst{{{value:.3f}}}"
    elif second_best_val is not None and abs(value - second_best_val) < 1e-6:  # 次最小值
        return f"\\subbst{{{value:.3f}}}"
    else:
        return f"{value:.3f}"

# 找到最佳和次最佳值
best_values = find_best_and_second_best(df_meta_avg)

contents = []

for model in ['TimeBridge', 'Fredformer']:
    line = r"\multirow{4}{*}{{\rotatebox{90}{\scaleb{" + model + r"}}}}"
    contents.append(line)
    for i, data_id in enumerate(['ETTm1', 'ETTh1', 'ECL', 'Weather']):
        _df = df_meta_avg[(df_meta_avg.model == model) & (df_meta_avg.data_id == data_id)]
        line = f"& {data_id} "
        
        # 获取当前model和data_id的最佳值信息
        best_info = best_values[model][data_id]
        
        for row in _df.itertuples():
            # 格式化MSE
            mse_formatted = format_value(row.mse, best_info['mse_best'], best_info['mse_second_best'])
            # 格式化MAE
            mae_formatted = format_value(row.mae, best_info['mae_best'], best_info['mae_second_best'])
            
            line += f"& {mse_formatted} & {mae_formatted} "
        line += r"\\"
        contents.append(line)
    if model != 'Fredformer':
        contents.append(r"\midrule" + "\n")

print('\n'.join(contents))


\multirow{4}{*}{{\rotatebox{90}{\scaleb{TimeBridge}}}}
& ETTm1 & \bst{0.383} & \bst{0.398} & 0.388 & 0.400 & \subbst{0.385} & 0.400 & 0.387 & \subbst{0.399} & 0.387 & 0.400 & 0.387 & 0.400 \\
& ETTh1 & \bst{0.433} & \subbst{0.437} & 0.441 & 0.439 & 0.438 & \bst{0.437} & 0.441 & 0.440 & \subbst{0.437} & 0.438 & 0.442 & 0.440 \\
& ECL & \bst{0.172} & \subbst{0.267} & 0.177 & 0.272 & 0.174 & 0.269 & \subbst{0.172} & \bst{0.266} & 0.176 & 0.271 & 0.176 & 0.271 \\
& Weather & \bst{0.248} & \bst{0.275} & 0.251 & \subbst{0.276} & 0.253 & 0.278 & \subbst{0.250} & 0.276 & 0.253 & 0.277 & 0.252 & 0.277 \\
\midrule

\multirow{4}{*}{{\rotatebox{90}{\scaleb{Fredformer}}}}
& ETTm1 & \bst{0.379} & \bst{0.395} & 0.386 & 0.397 & \subbst{0.380} & \subbst{0.395} & 0.385 & 0.397 & 0.385 & 0.397 & 0.387 & 0.398 \\
& ETTh1 & \bst{0.429} & \bst{0.431} & 0.445 & 0.435 & \subbst{0.437} & \subbst{0.432} & 0.444 & 0.435 & 0.444 & 0.435 & 0.447 & 0.434 \\
& ECL & \bst{0.183} & \bst{0.275} & 0.187 & 0.280 & 0.188 

## write two column table

In [7]:
dfm = df_meta.copy()
dfm['meta_method'].replace({'None': 'DF', 'FOMAML': 'QDF'}, inplace=True)
dfm[dfm.data_id == 'ECL']

/tmp/ipykernel_1684406/3007824618.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfm['meta_method'].replace({'None': 'DF', 'FOMAML': 'QDF'}, inplace=True)
/tmp/ipykernel_1684406/3007824618.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  dfm['meta_method'].replace({'None': 'DF', 'FOMAML': 'QDF'}, inplace=Tru

,model,pred_len,data_id,meta_method,mse,mae
0,TQNet,96,ECL,DF,0.143456,0.237354
1,TQNet,192,ECL,DF,0.160641,0.252081
2,TQNet,336,ECL,DF,0.178238,0.269703
3,TQNet,720,ECL,DF,0.217629,0.302696
48,TQNet,Avg,ECL,DF,0.174991,0.265458
12,TQNet,96,ECL,iMAML,0.135219,0.229624
17,TQNet,192,ECL,iMAML,0.153721,0.245653
22,TQNet,336,ECL,iMAML,0.170256,0.263047
27,TQNet,720,ECL,iMAML,0.204781,0.292530
49,TQNet,Avg,ECL,iMAML,0.165994,0.257713


In [12]:
contents = []
data_id = 'ECL'

for method in ['QDF', 'iMAML', 'MAML', 'MAML++', 'Reptile', 'DF']:
    line = method + " "
    _df = dfm[(dfm.meta_method == method) & (dfm.data_id == data_id)]
    print(_df)
    for row in _df.itertuples():
        line += f"& {row.mse:.3f} & {row.mae:.3f} "
    line += r"\\"
    contents.append(line)

print('\n'.join(contents))

    model pred_len data_id meta_method       mse       mae
8   TQNet       96     ECL         QDF  0.134702  0.228727
13  TQNet      192     ECL         QDF  0.153000  0.244973
18  TQNet      336     ECL         QDF  0.169068  0.262404
23  TQNet      720     ECL         QDF  0.201599  0.290306
53  TQNet      Avg     ECL         QDF  0.164592  0.256603
    model pred_len data_id meta_method       mse       mae
12  TQNet       96     ECL       iMAML  0.135219  0.229624
17  TQNet      192     ECL       iMAML  0.153721  0.245653
22  TQNet      336     ECL       iMAML  0.170256  0.263047
27  TQNet      720     ECL       iMAML  0.204781  0.292530
49  TQNet      Avg     ECL       iMAML  0.165994  0.257713
    model pred_len data_id meta_method       mse       mae
9   TQNet       96     ECL        MAML  0.135512  0.229762
14  TQNet      192     ECL        MAML  0.153826  0.245854
19  TQNet      336     ECL        MAML  0.169835  0.262807
24  TQNet      720     ECL        MAML  0.205335  0.2933

In [18]:
import pandas as pd

contents = []
data_id = 'ECL'

# 找到每个pred_len下mse和mae的最佳和次最佳方法
best_methods = {}

for pred_len in [96, 192, 336, 720, 'Avg']:
    _df_pred = dfm[(dfm.pred_len == pred_len) & (dfm.data_id == data_id)]
    
    # 对mse排序，获取最佳和次最佳方法
    mse_sorted = _df_pred.sort_values('mse')
    best_mse_method = mse_sorted.iloc[0]['meta_method']
    second_best_mse_method = mse_sorted.iloc[1]['meta_method']
    
    # 对mae排序，获取最佳和次最佳方法
    mae_sorted = _df_pred.sort_values('mae')
    best_mae_method = mae_sorted.iloc[0]['meta_method']
    second_best_mae_method = mae_sorted.iloc[1]['meta_method']
    
    best_methods[pred_len] = {
        'best_mse': best_mse_method,
        'second_best_mse': second_best_mse_method,
        'best_mae': best_mae_method,
        'second_best_mae': second_best_mae_method
    }

# 生成latex表格内容
for method in ['QDF', 'iMAML', 'MAML', 'MAML++', 'Reptile', 'DF']:
    line = method + " "
    _df = dfm[(dfm.meta_method == method) & (dfm.data_id == data_id)]
    
    for row in _df.itertuples():
        pred_len = row.pred_len
        mse_val = row.mse
        mae_val = row.mae
        
        # 检查mse是否为最佳或次最佳
        if method == best_methods[pred_len]['best_mse']:
            mse_str = f"\\bst{{{mse_val:.3f}}}"
        elif method == best_methods[pred_len]['second_best_mse']:
            mse_str = f"\\subbst{{{mse_val:.3f}}}"
        else:
            mse_str = f"{mse_val:.3f}"
        
        # 检查mae是否为最佳或次最佳
        if method == best_methods[pred_len]['best_mae']:
            mae_str = f"\\bst{{{mae_val:.3f}}}"
        elif method == best_methods[pred_len]['second_best_mae']:
            mae_str = f"\\subbst{{{mae_val:.3f}}}"
        else:
            mae_str = f"{mae_val:.3f}"
        
        line += f"&& {mse_str} & {mae_str} "
    
    line += r"\\"
    contents.append(line)

print('\n'.join(contents))


QDF && \bst{0.135} & \bst{0.229} && \bst{0.153} & \bst{0.245} && \bst{0.169} & \bst{0.262} && \bst{0.202} & \bst{0.290} && \bst{0.165} & \bst{0.257} \\
iMAML && 0.135 & 0.230 && \subbst{0.154} & \subbst{0.246} && 0.170 & 0.263 && 0.205 & 0.293 && 0.166 & 0.258 \\
MAML && 0.136 & 0.230 && 0.154 & 0.246 && 0.170 & 0.263 && 0.205 & 0.293 && 0.166 & 0.258 \\
MAML++ && \subbst{0.135} & \subbst{0.229} && 0.154 & 0.246 && \subbst{0.170} & \subbst{0.263} && \subbst{0.204} & \subbst{0.292} && \subbst{0.166} & \subbst{0.257} \\
Reptile && 0.136 & 0.230 && 0.155 & 0.247 && 0.171 & 0.264 && 0.206 & 0.294 && 0.167 & 0.259 \\
DF && 0.143 & 0.237 && 0.161 & 0.252 && 0.178 & 0.270 && 0.218 & 0.303 && 0.175 & 0.265 \\


In [16]:
import pandas as pd

dfm = df_meta.copy()
dfm['meta_method'].replace({'None': 'DF', 'FOMAML': 'QDF'}, inplace=True)
dfm[dfm.data_id == 'ECL']

contents = []
data_id = 'ECL'

# 找到每个pred_len下mse和mae的最佳和次最佳方法
best_methods = {}

for pred_len in [96, 192, 336, 720, 'Avg']:
    _df_pred = dfm[(dfm.pred_len == pred_len) & (dfm.data_id == data_id)]
    
    # 对mse排序，获取最佳和次最佳方法
    mse_sorted = _df_pred.sort_values('mse')
    best_mse_method = mse_sorted.iloc[0]['meta_method']
    second_best_mse_method = mse_sorted.iloc[1]['meta_method']
    
    # 对mae排序，获取最佳和次最佳方法
    mae_sorted = _df_pred.sort_values('mae')
    best_mae_method = mae_sorted.iloc[0]['meta_method']
    second_best_mae_method = mae_sorted.iloc[1]['meta_method']
    
    best_methods[pred_len] = {
        'best_mse': best_mse_method,
        'second_best_mse': second_best_mse_method,
        'best_mae': best_mae_method,
        'second_best_mae': second_best_mae_method
    }

# 获取DF方法的baseline值
df_baseline = {}
_df_baseline = dfm[(dfm.meta_method == 'DF') & (dfm.data_id == data_id)]
for row in _df_baseline.itertuples():
    df_baseline[row.pred_len] = {
        'mse': row.mse,
        'mae': row.mae
    }

# 生成latex表格内容
for method in ['DF', 'iMAML', 'MAML', 'MAML++', 'Reptile', 'QDF']:
    line = method + " "
    _df = dfm[(dfm.meta_method == method) & (dfm.data_id == data_id)]
    
    for row in _df.itertuples():
        pred_len = row.pred_len
        if pred_len == 'Avg':
            continue
        mse_val = row.mse
        mae_val = row.mae
        
        # 计算相对于DF的提升百分比
        if method != 'DF':
            baseline_mse = df_baseline[pred_len]['mse']
            baseline_mae = df_baseline[pred_len]['mae']
            
            # 计算相对提升: (baseline - current) / baseline * 100
            mse_improvement = (baseline_mse - mse_val) / baseline_mse * 100
            mae_improvement = (baseline_mae - mae_val) / baseline_mae * 100
            
            # 确定箭头方向和格式化
            mse_arrow = "\\downarrow" if mse_improvement > 0 else "\\uparrow"
            mae_arrow = "\\downarrow" if mae_improvement > 0 else "\\uparrow"
            
            mse_suffix = f"$_{{\\textcolor{{c1}}{{{abs(mse_improvement):.2f}\\%{mse_arrow}}}}}$"
            mae_suffix = f"$_{{\\textcolor{{c1}}{{{abs(mae_improvement):.2f}\\%{mae_arrow}}}}}$"
        else:
            # DF方法不显示相对提升
            mse_suffix = ""
            mae_suffix = ""
        
        # 检查mse是否为最佳或次最佳，并添加相对提升
        if method == best_methods[pred_len]['best_mse']:
            mse_str = f"\\bst{{{mse_val:.3f}}}{mse_suffix}"
        elif method == best_methods[pred_len]['second_best_mse']:
            mse_str = f"\\subbst{{{mse_val:.3f}}}{mse_suffix}"
        else:
            mse_str = f"{mse_val:.3f}{mse_suffix}"
        
        # 检查mae是否为最佳或次最佳，并添加相对提升
        if method == best_methods[pred_len]['best_mae']:
            mae_str = f"\\bst{{{mae_val:.3f}}}{mae_suffix}"
        elif method == best_methods[pred_len]['second_best_mae']:
            mae_str = f"\\subbst{{{mae_val:.3f}}}{mae_suffix}"
        else:
            mae_str = f"{mae_val:.3f}{mae_suffix}"
        
        line += f"& {mse_str} & {mae_str} "
    
    line += r"\\"
    contents.append(line)

print('\n'.join(contents))


DF & 0.143 & 0.237 & 0.161 & 0.252 & 0.178 & 0.270 & 0.218 & 0.303 \\
iMAML & 0.135$_{\textcolor{c1}{5.74\%\downarrow}}$ & 0.230$_{\textcolor{c1}{3.26\%\downarrow}}$ & \subbst{0.154}$_{\textcolor{c1}{4.31\%\downarrow}}$ & \subbst{0.246}$_{\textcolor{c1}{2.55\%\downarrow}}$ & 0.170$_{\textcolor{c1}{4.48\%\downarrow}}$ & 0.263$_{\textcolor{c1}{2.47\%\downarrow}}$ & 0.205$_{\textcolor{c1}{5.90\%\downarrow}}$ & 0.293$_{\textcolor{c1}{3.36\%\downarrow}}$ \\
MAML & 0.136$_{\textcolor{c1}{5.54\%\downarrow}}$ & 0.230$_{\textcolor{c1}{3.20\%\downarrow}}$ & 0.154$_{\textcolor{c1}{4.24\%\downarrow}}$ & 0.246$_{\textcolor{c1}{2.47\%\downarrow}}$ & 0.170$_{\textcolor{c1}{4.71\%\downarrow}}$ & 0.263$_{\textcolor{c1}{2.56\%\downarrow}}$ & 0.205$_{\textcolor{c1}{5.65\%\downarrow}}$ & 0.293$_{\textcolor{c1}{3.09\%\downarrow}}$ \\
MAML++ & \subbst{0.135}$_{\textcolor{c1}{5.76\%\downarrow}}$ & \subbst{0.229}$_{\textcolor{c1}{3.33\%\downarrow}}$ & 0.154$_{\textcolor{c1}{4.22\%\downarrow}}$ & 0.246$_{\text

/tmp/ipykernel_1705335/2013025955.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfm['meta_method'].replace({'None': 'DF', 'FOMAML': 'QDF'}, inplace=True)
/tmp/ipykernel_1705335/2013025955.py:4: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  dfm['meta_method'].replace({'None': 'DF', 'FOMAML': 'QDF'}, inplace=Tru